# 문항 1 : 순차 · 스레드 · 프로세스 속도 비교

## 같은 작업을 세 가지 방식으로 실행해 소요시간을 비교하시오.

대상은 로컬 Flask 서버를 활용하세요.

세 방식

1. 순차 반복문

2. multiprocessing.dummy.Pool(n).map (스레드)

3. multiprocessing.Pool(n).map (프로세스)

워커 수 1 · 3 · 5 · 10 을 바꿔가며 측정해 표로 정리

측정 결과를 보고 두 방식의 차이가 어디서 오는지 2~3줄로 설명할 것 (힌트: 대기 시간, 풀 생성 비용 · 메모리 · 직렬화 제약도 함께 생각해볼 것)

In [ ]:
# bench_server.py

from flask import Flask, jsonify
import time, random

app = Flask(__name__)

@app.get("/item/<int:n>")
def item(n):
    time.sleep(random.uniform(0.3, 0.6))
    return jsonify({"id": n, "name": f"item-{n}"})

if __name__ == "__main__":
    app.run(port=5000, threaded=True)

In [ ]:
# assignment_sequential(순차 반복문)

import time
import re
import requests

def urllib_extract():
    # news_오디세이.csv 파일에서 URL 추출하기
    with open("news_오디세이.csv", "r", encoding="utf-8") as f:
        content = f.read()
    return re.findall(r'https?://[^\s,\"\']+', content)

def fetch_item(args):
    idx, _ = args
    target_url = f"http://127.0.0.1:5000/item/{idx}"
    try:
        response = requests.get(target_url, timeout=3)
        return response.json()
    except Exception as e:
        return {"error": str(e)}

if __name__ == "__main__":
    urls = urllib_extract()
    print(f"추출된 URL 개수: {len(urls)}개")
    
    start_time = time.time()
    
    results = []
    for i, url in enumerate(urls):
        res = fetch_item((i, url))
        results.append(res)
        
    end_time = time.time()
    print(f"[순차 반복문] 총 소요 시간: {end_time - start_time:.4f} 초")

    # 추출된 URL 50개
    # [순차 반복문] 총 소요 시간: 23.1767 초

In [ ]:
# assignment_thread(멀티스레드)

import time
import re
import requests
from multiprocessing.dummy import Pool

def urllib_extract():
    # news_오디세이.csv로 연동
    with open("news_오디세이.csv", "r", encoding="utf-8") as f:
        content = f.read()
    return re.findall(r'https?://[^\s,\"\']+', content)

def fetch_item(args):
    idx, _ = args
    target_url = f"http://127.0.0.1:5000/item/{idx}"
    try:
        response = requests.get(target_url, timeout=3)
        return response.json()
    except Exception as e:
        return {"error": str(e)}

if __name__ == "__main__":
    urls = urllib_extract()
    worker_sizes = [1, 3, 5, 10]
    
    print(f"추출된 URL 개수: {len(urls)}개")
    print("-" * 50)
    print(" [multiprocessing.dummy.Pool (스레드) 측정 시작] ")
    print("-" * 50)
    
    for n in worker_sizes:
        tasks = list(enumerate(urls))
        start_time = time.time()
        
        pool = Pool(n)
        results = pool.map(fetch_item, tasks)
        pool.close()
        pool.join()
        
        end_time = time.time()
        print(f"워커 수 {n:2d}개 일 때 소요 시간: {end_time - start_time:.4f} 초")
    print("-" * 50)

    # 추출된 URL 50개
    #  [multiprocessing.dummy.Pool (스레드) 측정 시작] 
    # --------------------------------------------------
    # 워커 수  1개 일 때 소요 시간: 24.1575 초
    # 워커 수  3개 일 때 소요 시간: 8.9683 초
    # 워커 수  5개 일 때 소요 시간: 5.1719 초
    # 워커 수 10개 일 때 소요 시간: 2.6959 초

    # 멀티스레드 방식 : 동일한 메모리를 공유하며 구동됨
    # 생성비용이 거의 없고, 자원 낭비가 적어 병렬 효율 극대화
    # 워커 수가 늘어날수록 프로세스 방식보다 짧게 소요됨
    # 네트워크 요청(대기시간위주) 작업에서는 프로세스보다 유리


In [ ]:
# assignment_process(멀티프로세스)

import time
import re
import requests
from multiprocessing import Pool

def urllib_extract():
    # news_오디세이.csv로 연동
    with open("news_오디세이.csv", "r", encoding="utf-8") as f:
        content = f.read()
    return re.findall(r'https?://[^\s,\"\']+', content)

def fetch_item(args):
    idx, _ = args
    target_url = f"http://127.0.0.1:5000/item/{idx}"
    try:
        response = requests.get(target_url, timeout=3)
        return response.json()
    except Exception as e:
        return {"error": str(e)}

if __name__ == "__main__":
    urls = urllib_extract()
    worker_sizes = [1, 3, 5, 10]
    
    print(f"추출된 URL 개수: {len(urls)}개")
    print("-" * 50)
    print(" [multiprocessing.Pool (프로세스) 측정 시작] ")
    print("-" * 50)
    
    for n in worker_sizes:
        tasks = list(enumerate(urls))
        start_time = time.time()
        
        pool = Pool(n)
        results = pool.map(fetch_item, tasks)
        pool.close()
        pool.join()
        
        end_time = time.time()
        print(f"워커 수 {n:2d}개 일 때 소요 시간: {end_time - start_time:.4f} 초")
    print("-" * 50)

    # 추출된 URL 50개
    #  [multiprocessing.Pool (프로세스) 측정 시작]  
    # --------------------------------------------------
    # 워커 수  1개 일 때 소요 시간: 23.4755 초
    # 워커 수  3개 일 때 소요 시간: 9.1817 초
    # 워커 수  5개 일 때 소요 시간: 5.2463 초
    # 워커 수 10개 일 때 소요 시간: 3.3140 초

    # 멀티프로세스 방식 : 독립된 메모리 공간 생성(복사)
    # 풀 생성 비용(오버헤드)이 크고,
    # 데이터를 복사하는 직렬화 제약으로 실행속도가 느림

# 문항 2 : Scrapy 포팅 또는 httpx 비동기 전환 (택일)

## B. httpx 비동기 전환

A 대상(quotes.toscrape.com)을 httpx.AsyncClient + asyncio.gather 로 다시 작성할 것

asyncio.Semaphore 와 httpx.Limits 로 동시 수를 이중 제한할 것

time.sleep 대신 await asyncio.sleep 을 쓸 것

return_exceptions=True 로 부분 실패를 허용할 것

소요시간을 측정해볼 것

In [ ]:
import asyncio
import time
import httpx
from bs4 import BeautifulSoup

# [조건 2] Semaphore로 전체 동시 요청 수 제한
SEMAPHORE_LIMIT = 5
sem = asyncio.Semaphore(SEMAPHORE_LIMIT)

# [조건 2] httpx.Limits로 연결 풀 크기 및 호스트당 동시 요청 수 이중 제한
# 전체 최대 10개 연결, 호스트당 최대 5개 연결 유지
limits = httpx.Limits(max_connections=10, max_keepalive_connections=5)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

async def fetch_page(client, page_num):
    url = f"https://quotes.toscrape.com/page/{page_num}/"
    
    async with sem:
        try:
            response = await client.get(url, headers=HEADERS, timeout=10.0)
            
            # [조건 3] time.sleep 대신 await asyncio.sleep
            await asyncio.sleep(0.2)
            
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')
                quotes = soup.find_all('div', class_='quote')
                print(f"[성공] Page {page_num}: {len(quotes)}개의 명언 수집됨")
                return len(quotes)
            else:
                print(f"[실패] Page {page_num}: 상태 코드 {response.status_code}")
                return 0
        except Exception as e:
            print(f"[에러] Page {page_num}: {str(e)}")
            raise e

async def main():
    pages = list(range(1, 11))
    
    # [조건 5] 소요시간 측정 시작
    start_time = time.time()
    
    # [조건 1] httpx.AsyncClient 생성
    async with httpx.AsyncClient(limits=limits) as client:
        # [조건 1] asyncio.gather 로 비동기 태스크 묶기
        # [조건 4] return_exceptions=True 로 부분 실패를 허용할 것
        tasks = [fetch_page(client, p) for p in pages]
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
    end_time = time.time()
    
    print("-" * 50)
    print(f"작업 완료 결과 목록: {results}")
    print(f"[httpx 비동기 전환] 총 소요 시간: {end_time - start_time:.4f} 초")
    print("-" * 50)

if __name__ == "__main__":
    import sys
    if sys.platform == 'win32':
        asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
    asyncio.run(main())

    # --------------------------------------------------
    # 작업 완료 결과 목록: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
    # [httpx 비동기 전환] 총 소요 시간: 2.0117 초
    # --------------------------------------------------
